<a href="https://colab.research.google.com/github/pusalaspreetham-code/amazon-review-analyzer/blob/main/data_prepocessing/sentiment_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import libraries and load the dataset

In [6]:
import pandas as pd
import numpy as np

# Load preprocessed dataset
df = pd.read_csv("/content/amazon_reviews_cleaned.csv")

# Display dataset information
print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (11462, 20)


,review_id,product_id,product_name,category,product_url,price,original_price,product_rating,total_product_reviews,five_star_review_count,four_star_review_count,three_star_review_count,two_star_review_count,one_star_review_count,review_title,review_text,review_rating,helpful_votes,review_age,discount_percentage
0,RKSVOZF894JFW,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,214,91,29,76,Best shower gel,Love this shower gel feels fresh in every shower,5.0,0,1 year ago,61.78
1,R1ZNY7WGCQTRY4,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,214,91,29,76,Doubt its original as quality and scent not same,Quality deteriorated scent is almost negligibl...,2.0,2,3 years ago,61.78
2,RK2R8UNJ28858,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,214,91,29,76,Average product,Soap remains on body even after thorough wash,3.0,1,3 years ago,61.78
3,R1PSRBJB5CR0XE,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,214,91,29,76,Awesome fragrance,Fragrance is really nice and refreshing also e...,4.0,1,2 years ago,61.78
4,R3FK9S0MB6SRS4,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,214,91,29,76,The nozzle was open so liquid soap leaked,The nozzle was not correctly and liquid soap w...,1.0,1,2 years ago,61.78


Check review text

In [7]:
df["review_text"].head(5)

,review_text
0,Love this shower gel feels fresh in every shower
1,Quality deteriorated scent is almost negligibl...
2,Soap remains on body even after thorough wash
3,Fragrance is really nice and refreshing also e...
4,The nozzle was not correctly and liquid soap w...


Roberta Model

In [8]:
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix
# Load pretrained sentiment analysis model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

print("Sentiment model loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model loaded successfully!


Test on 5 sample reviews

In [9]:
sample_reviews = df["review_text"].head(5).tolist()
sentiment_results = sentiment_model(sample_reviews)

for review, prediction in zip(sample_reviews, sentiment_results):
    print("Review:", review)
    print("Predicted Sentiment:", prediction["label"])
    print("Confidence:", round(prediction["score"] * 100, 2), "%")
    print("-" * 60)

Review: Love this shower gel feels fresh in every shower
Predicted Sentiment: positive
Confidence: 97.64 %
------------------------------------------------------------
Review: Quality deteriorated scent is almost negligible.....using for last 3years hence easily notice the difference
Predicted Sentiment: negative
Confidence: 64.54 %
------------------------------------------------------------
Review: Soap remains on body even after thorough wash
Predicted Sentiment: neutral
Confidence: 76.75 %
------------------------------------------------------------
Review: Fragrance is really nice and refreshing also ease to use also overall nice product go for it and Product is safe but Amazon packing is worst
Predicted Sentiment: positive
Confidence: 90.94 %
------------------------------------------------------------
Review: The nozzle was not correctly and liquid soap was leaked
Predicted Sentiment: negative
Confidence: 79.5 %
------------------------------------------------------------


Testing for 50 reviews


In [10]:
# Select 50 reviews
sample_reviews = df["review_text"].head(50).tolist()

# Predict sentiment
predictions = sentiment_model(sample_reviews)

# Create a result dataframe
test_results = pd.DataFrame({
    "review_text": sample_reviews,
    "sentiment": [p["label"] for p in predictions],
    "confidence": [round(p["score"] * 100, 2) for p in predictions]
})

test_results.head(50)

,review_text,sentiment,confidence
0,Love this shower gel feels fresh in every shower,positive,97.64
1,Quality deteriorated scent is almost negligibl...,negative,64.54
2,Soap remains on body even after thorough wash,neutral,76.75
3,Fragrance is really nice and refreshing also e...,positive,90.94
4,The nozzle was not correctly and liquid soap w...,negative,79.50
5,I expected a predominance of citrus notes. A g...,positive,79.84
6,Received old stock as the manufacturing date w...,neutral,49.05
7,The smell is great but it does not feel fresh ...,negative,59.70
8,It's ok,positive,49.66
9,I am very happy lo🫶🫶🫶 ❤️❤️❤️ Amazon,positive,98.90


Perfoming on total dataset

In [11]:
# Select genuine Amazon reviews
reviews = df["review_text"].astype(str).tolist()

# Predict sentiment for all reviews
sentiment_results = sentiment_model(
    reviews,
    batch_size=32,
    truncation=True,
    max_length=512
)

print("Total reviews processed:", len(sentiment_results))

Total reviews processed: 11462


Store predictions and confidence scores

In [12]:
result_df = df.copy()

result_df["predicted_sentiment"] = [
    result["label"].lower()
    for result in sentiment_results
]

result_df["confidence"] = [
    result["score"] * 100
    for result in sentiment_results
]

display(result_df.head())

,review_id,product_id,product_name,category,product_url,price,original_price,product_rating,total_product_reviews,five_star_review_count,...,two_star_review_count,one_star_review_count,review_title,review_text,review_rating,helpful_votes,review_age,discount_percentage,predicted_sentiment,confidence
0,RKSVOZF894JFW,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,...,29,76,Best shower gel,Love this shower gel feels fresh in every shower,5.0,0,1 year ago,61.78,positive,97.643799
1,R1ZNY7WGCQTRY4,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,...,29,76,Doubt its original as quality and scent not same,Quality deteriorated scent is almost negligibl...,2.0,2,3 years ago,61.78,negative,64.535159
2,RK2R8UNJ28858,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,...,29,76,Average product,Soap remains on body even after thorough wash,3.0,1,3 years ago,61.78,neutral,76.746076
3,R1PSRBJB5CR0XE,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,...,29,76,Awesome fragrance,Fragrance is really nice and refreshing also e...,4.0,1,2 years ago,61.78,positive,90.944135
4,R3FK9S0MB6SRS4,B00791CQ5K,"Palmolive Aroma Morning Tonic Body Wash, 250ml...",Beauty > Bath & Body > Cleansers > Body Wash Gels,https://amzn.in/d/0iydsUco,99.0,259.0,4.4,8792,465,...,29,76,The nozzle was open so liquid soap leaked,The nozzle was not correctly and liquid soap w...,1.0,1,2 years ago,61.78,negative,79.497749


Analysis of model

In [13]:
print(result_df["predicted_sentiment"].value_counts())

print("\nAverage confidence:")
print(result_df.groupby("predicted_sentiment")["confidence"].mean())

predicted_sentiment
positive    4927
negative    4777
neutral     1758
Name: count, dtype: int64

Average confidence:
predicted_sentiment
negative    80.538346
neutral     63.303507
positive    84.472941
Name: confidence, dtype: float64


Seprating based on confidence

In [14]:
high_confidence = result_df[
    result_df["confidence"] >= 85
].copy()

low_confidence = result_df[
    result_df["confidence"] < 85
].copy()

print("High-confidence reviews:", len(high_confidence))
print("Low-confidence reviews:", len(low_confidence))

High-confidence reviews: 5591
Low-confidence reviews: 5871


In [18]:
def validate_sentiment(row):
    rating = row["review_rating"]
    prediction = row["predicted_sentiment"]

    if rating >= 4 and prediction == "positive":
        return "positive"

    elif rating <= 2 and prediction == "negative":
        return "negative"

    elif rating == 3 and prediction == "neutral":
        return "neutral"

    else:
        return "needs_review"


result_df["final_sentiment"] = result_df.apply(
    validate_sentiment, axis=1
)

# Separate accepted and mismatched reviews
accepted_reviews = result_df[
    result_df["final_sentiment"] != "needs_review"
].copy()

review_needed = result_df[
    result_df["final_sentiment"] == "needs_review"
].copy()

print("Total reviews:", len(result_df))
print("Accepted reviews:", len(accepted_reviews))
print("Needs review:", len(review_needed))

print("\nAccepted sentiment distribution:")
print(accepted_reviews["final_sentiment"].value_counts())

Total reviews: 11462
Accepted reviews: 8233
Needs review: 3229

Accepted sentiment distribution:
final_sentiment
positive    4153
negative    3564
neutral      516
Name: count, dtype: int64


In [19]:
# Select 100 reviews from each rating group
positive_sample = result_df[
    result_df["review_rating"].isin([4, 5])
].sample(n=100, random_state=42)

negative_sample = result_df[
    result_df["review_rating"].isin([1, 2])
].sample(n=100, random_state=42)

neutral_sample = result_df[
    result_df["review_rating"] == 3
].sample(n=100, random_state=42)

manual_sample = pd.concat([
    positive_sample,
    negative_sample,
    neutral_sample
])

manual_sample = manual_sample.sample(
    frac=1,
    random_state=42
)

manual_sample["actual_sentiment"] = ""

manual_sample.to_csv(
    "manual_labelling_300.csv",
    index=False
)

print("Manual labelling sample created:", len(manual_sample))

Manual labelling sample created: 300


Prepare training and testing datasets

In [20]:
import pandas as pd

# Load manually labelled test dataset
test_df = pd.read_csv("manual_labelling_300_labeled.csv")

# Remove manually labelled test reviews from training data
test_ids = test_df["review_id"].tolist()

train_df = accepted_reviews[
    ~accepted_reviews["review_id"].isin(test_ids)
].copy()

# Prepare training and testing data
X_train = train_df["review_text"]
y_train = train_df["final_sentiment"]

X_test = test_df["review_text"]
y_test = test_df["actual_sentiment"]

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 8036
Testing records: 300


In [21]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Prepare text and labels
X_train = train_df["review_text"].fillna("")
y_train = train_df["final_sentiment"]

X_test = test_df["review_text"].fillna("")
y_test = test_df["actual_sentiment"]

# Define models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced"
    ),

    "Naive Bayes": MultinomialNB()
}

results = []

# Train and evaluate each model
for name, model in models.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=30000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, average="macro", zero_division=0
    )
    recall = recall_score(
        y_test, predictions, average="macro", zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, average="macro", zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Macro F1": f1
    })

    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    print(classification_report(
        y_test, predictions, zero_division=0
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(
        y_test,
        predictions,
        labels=["negative", "neutral", "positive"]
    ))

    # Keep trained model
    if name == "Logistic Regression":
        lr_model = pipeline
    elif name == "Linear SVM":
        svm_model = pipeline
    else:
        nb_model = pipeline

# Compare models
results_df = pd.DataFrame(results)

print("\nFINAL MODEL COMPARISON")
display(
    results_df.sort_values(
        by="Macro F1",
        ascending=False
    ).reset_index(drop=True)
)


Logistic Regression
              precision    recall  f1-score   support

    negative       0.62      0.57      0.59       148
     neutral       0.21      0.23      0.22        47
    positive       0.55      0.57      0.56       105

    accuracy                           0.52       300
   macro avg       0.46      0.46      0.46       300
weighted avg       0.53      0.52      0.52       300

Confusion Matrix:
[[85 27 36]
 [22 11 14]
 [31 14 60]]

Linear SVM
              precision    recall  f1-score   support

    negative       0.62      0.61      0.62       148
     neutral       0.19      0.13      0.15        47
    positive       0.50      0.59      0.54       105

    accuracy                           0.53       300
   macro avg       0.44      0.44      0.44       300
weighted avg       0.51      0.53      0.52       300

Confusion Matrix:
[[91 14 43]
 [23  6 18]
 [32 11 62]]

Naive Bayes
              precision    recall  f1-score   support

    negative       0.58    

,Model,Accuracy,Precision,Recall,Macro F1
0,Logistic Regression,0.52,0.457645,0.459932,0.458256
1,Linear SVM,0.53,0.440300,0.444334,0.438918
2,Naive Bayes,0.54,0.577778,0.433575,0.411168


In [22]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

param_grid = {
    "classifier__C": [0.1, 0.5, 1, 2, 5, 10]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best Cross-validation Macro F1:", grid.best_score_)

Best Parameters: {'classifier__C': 5}
Best Cross-validation Macro F1: 0.7614320979730739


In [23]:
best_model = grid.best_estimator_

test_predictions = best_model.predict(X_test)

print("Test Accuracy:",
      accuracy_score(y_test, test_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    test_predictions,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    test_predictions,
    labels=["negative", "neutral", "positive"]
))

Test Accuracy: 0.5233333333333333

Classification Report:
              precision    recall  f1-score   support

    negative       0.63      0.59      0.61       148
     neutral       0.20      0.19      0.20        47
    positive       0.52      0.57      0.54       105

    accuracy                           0.52       300
   macro avg       0.45      0.45      0.45       300
weighted avg       0.52      0.52      0.52       300


Confusion Matrix:
[[88 21 39]
 [21  9 17]
 [31 14 60]]


In [24]:
from sklearn.metrics import classification_report

roberta_predictions = test_df["predicted_sentiment"]

print(classification_report(
    test_df["actual_sentiment"],
    roberta_predictions,
    zero_division=0
))

              precision    recall  f1-score   support

    negative       0.64      0.61      0.62       148
     neutral       0.28      0.32      0.30        47
    positive       0.58      0.59      0.59       105

    accuracy                           0.56       300
   macro avg       0.50      0.51      0.50       300
weighted avg       0.57      0.56      0.56       300

